In [0]:
pip install findspark

In [0]:
import findspark, pyspark
from pyspark.sql import SparkSession
findspark.init()
spark = SparkSession.builder.appName('logistic_regression').getOrCreate()

In [0]:
from pyspark.ml.feature import RFormula

In [0]:
abs_path = "/Volumes/workspace/ml_with_spark/raw_data/Churn.csv"

churn = spark.read.csv(abs_path, header=True, inferSchema=True, sep=';')
churn.show(5)

In [0]:
rformula = RFormula(formula='Exited ~ .', featuresCol='independente', labelCol='dependente')
churn_rf = rformula.fit(churn).transform(churn)
churn_rf.select('independente', 'dependente').show(5, truncate=False)

In [0]:
churn_treino, churn_teste = churn_rf.randomSplit([0.8, 0.2])
print(f'Quantidade de linhas no conjunto de treino: {churn_treino.count()}')
print(f'Quantidade de linhas no conjunto de teste: {churn_teste.count()}')


In [0]:
from pyspark.ml.classification import LogisticRegression

log_reg = LogisticRegression(
        featuresCol='independente',
        labelCol='dependente',
        maxIter=1000,
        regParam=0.08
    )

modelo = log_reg.fit(churn_treino)

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

previsao = modelo.transform(churn_teste)

avaliador_acc = MulticlassClassificationEvaluator(
    labelCol='dependente',
    predictionCol='prediction',
    metricName='accuracy'
)

acuracia = avaliador_acc.evaluate(previsao)

avaliador_auc = BinaryClassificationEvaluator(
    labelCol='dependente',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)

auc = avaliador_auc.evaluate(previsao)

avaliador_precision = MulticlassClassificationEvaluator(
    labelCol='dependente',
    predictionCol='prediction',
    metricName='weightedPrecision'
)

precision = avaliador_precision.evaluate(previsao)

avaliador_recall = MulticlassClassificationEvaluator(
    labelCol='dependente',
    predictionCol='prediction',
    metricName='weightedRecall'
)

recall = avaliador_recall.evaluate(previsao)



print(f"Acurácia: {acuracia}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"AUC: {auc}")